# B3 — single common fine reference (950,400 el) for every QoI, per Professor Rabczuk's second review

**Why this notebook exists, and how it differs from `B3_FineReference_Reevaluation.ipynb`**: that earlier notebook split the fine reference by QoI (950,400 el for stress/H1, 43,400 el for displacement), matching the submitted paper's own convention. Professor Rabczuk's own follow-up email asked for something more exacting this round: **one common 950,400-element reference for every relevant QoI** (displacement, H1 semi-norm, energy, reaction moment, regional stress, and the individual stress components), for every existing checkpoint — no retraining, evaluation only.

He also asked for something genuinely new: **how much of the gap between a coarse FEM solution and the fine reference is just mesh-discretization error, with no neural network involved at all** — i.e. solve FEM (not a neural operator) at 6,840 and 43,400 elements, compare each directly against the 950,400-element solution, and report that separately from any checkpoint's own error. This lets a reader see, side by side, how much of a checkpoint's 950,400-vs-training-mesh gap a perfect, infinitely-accurate neural operator would still show just from training-mesh discretization error alone.

**Checkpoints evaluated against the single 950,400-element reference** (all QoIs, including the per-component stress breakdown added 2026-10-30): PI baseline; items 2/3 attempts 1 and 2 (equilibrium-augmented); item 4's data-driven rel_l2 at N=50/200/400, Sobolev, and hybrid; item 1's mixed-stress attempts 1, 5, and 6 (both the implicit and direct stress paths each).

**FEM-vs-FEM discretization error**: the SAME held-out samples' 6,840-element (already in `dataset.h5`, no new solve) and 43,400-element solutions, each compared directly against the 950,400-element reference using the exact same QoI formulas as the neural-operator comparison above, so the two kinds of numbers are genuinely comparable.

In [ ]:
# =====================================================================
#  CELL -- Re-evaluate every existing B3 checkpoint against ONE common
#  950,400-element reference for every QoI (Prof. Rabczuk's 2nd email),
#  plus a FEM-vs-FEM discretization-error report (6,840 and 43,400
#  element solutions vs. the same 950,400-element reference).
# =====================================================================
import os
os.environ['JAX_PLATFORMS'] = 'cpu'

import subprocess
import sys
import time

_started = time.time()


def run(cmd):
    print('$', ' '.join(str(c) for c in cmd), flush=True)
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE,
                          stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end='', flush=True)
    p.wait()
    if p.returncode != 0:
        raise subprocess.CalledProcessError(p.returncode, cmd)


from google.colab import drive
if os.path.isdir('/content/drive/MyDrive'):
    print('Drive already mounted at /content/drive.')
else:
    drive.mount('/content/drive')

REPO = '/content/OMAR'
if not os.path.isdir(REPO):
    run(['git', 'clone', '-b', 'claude/claude-code-question-d307wp',
         'https://github.com/SUHIBAMRO/OMAR.git', REPO])
else:
    run(['git', '-C', REPO, 'fetch', 'origin', 'claude/claude-code-question-d307wp'])
    run(['git', '-C', REPO, 'checkout', 'claude/claude-code-question-d307wp'])
    run(['git', '-C', REPO, 'reset', '--hard', 'origin/claude/claude-code-question-d307wp'])

run([sys.executable, '-m', 'pip', 'install', '-q', 'pyvista<0.49'])
run([sys.executable, '-m', 'pip', 'install', '-q', 'torch-fem'])

WORK = f'{REPO}/Practical_Examples'
os.chdir(WORK)
sys.path.insert(0, WORK)

for _mod_name in list(sys.modules):
    if (_mod_name == 'omar_pfem' or _mod_name.startswith('omar_pfem.')
            or _mod_name == 'torchfem' or _mod_name.startswith('torchfem.')
            or _mod_name == 'pyvista' or _mod_name.startswith('pyvista.')):
        del sys.modules[_mod_name]

import torch
assert torch.cuda.is_available(), 'this cell needs a real GPU'
print('GPU:', torch.cuda.get_device_name(0))

import gc
for _attr in ('last_traceback', 'last_value', 'last_type'):
    if hasattr(sys, _attr):
        delattr(sys, _attr)
gc.collect()
torch.cuda.empty_cache()

from omar_pfem.evaluate_B3_qois_finer import (
    generate_fine_reference, evaluate_u_only_checkpoint_against_fine_reference,
    FINE_RESOLUTION_STRESS, FINE_RESOLUTION_DISP_ENERGY_REACTION,
)
from omar_pfem.evaluate_B3_mixed_stress_finer import evaluate_mixed_stress_checkpoint_against_fine_reference
from omar_pfem.evaluate_B3_fem_discretization_error import load_coarse_reference, compute_fem_discretization_errors

device = torch.device('cuda')
dtype = torch.float64
R = '/content/drive/MyDrive/pfem_run'
HOLDOUT = f'{R}/b3_dataset_clean_holdout/dataset.h5'
N_SAMPLES = 10

print(f'\n{"=" * 90}')
print(f'=== Generating the ONE common 950,400-element ({FINE_RESOLUTION_STRESS}) reference, {N_SAMPLES} samples ===')
t0 = time.time()
fine_ref = generate_fine_reference(HOLDOUT, FINE_RESOLUTION_STRESS, N_SAMPLES, device, dtype)
print(f'Fine reference generated in {time.time() - t0:.1f}s.')

U_ONLY_CHECKPOINTS = [
    ('PI baseline', f'{R}/b3_training_normalized/checkpoint_50000.pt'),
    ('Items 2/3, attempt 1 (equilibrium, w=1.0)', f'{R}/b3_equilibrium_augmented/checkpoint_50000.pt'),
    ('Items 2/3, attempt 2 (equilibrium, w=0.12)', f'{R}/b3_equilibrium_augmented_v2/checkpoint_50000.pt'),
    ('Item 4, DD rel_l2 @ N=50', f'{R}/b3_data_driven_rel_l2_N50/model_best.pt'),
    ('Item 4, DD rel_l2 @ N=200 (break-even baseline)', f'{R}/b3_data_driven_rel_l2/model_best.pt'),
    ('Item 4, DD rel_l2 @ N=400', f'{R}/b3_data_driven_rel_l2_N400/model_best.pt'),
    ('Item 4, DD sobolev', f'{R}/b3_data_driven_sobolev/model_best.pt'),
    ('Item 4, DD hybrid', f'{R}/b3_data_driven_hybrid/model_best.pt'),
]
MIXED_STRESS_CHECKPOINTS = [
    ('Item 1, attempt 1 (mixed-stress, scale=500)', f'{R}/b3_mixed_stress_training/checkpoint_50000.pt'),
    ('Item 1, attempt 5 (mixed-stress, local-refine)', f'{R}/b3_mixed_stress_local_refine/checkpoint_50000.pt'),
    ('Item 1, attempt 6 (mixed-stress, relative + local-refine)',
     f'{R}/b3_mixed_stress_local_refine_relative/checkpoint_50000.pt'),
]

results = {}
for name, ckpt in U_ONLY_CHECKPOINTS:
    print(f'\n{"=" * 90}')
    print(f'=== [vs. 950,400 el] {name} ===')
    if not os.path.exists(ckpt):
        print(f'  SKIPPED -- checkpoint not found at {ckpt}')
        continue
    results[name] = evaluate_u_only_checkpoint_against_fine_reference(ckpt, fine_ref, device, dtype)

for name, ckpt in MIXED_STRESS_CHECKPOINTS:
    print(f'\n{"=" * 90}')
    print(f'=== [vs. 950,400 el, implicit + direct] {name} ===')
    if not os.path.exists(ckpt):
        print(f'  SKIPPED -- checkpoint not found at {ckpt}')
        continue
    results[name] = evaluate_mixed_stress_checkpoint_against_fine_reference(ckpt, fine_ref, device, dtype)

print(f'\n{"=" * 90}')
print(f'=== Generating the 43,400-element ({FINE_RESOLUTION_DISP_ENERGY_REACTION}) solution, {N_SAMPLES} samples ===')
print('=== (needed only for the FEM-vs-FEM discretization-error report below) ===')
t0 = time.time()
mid_ref = generate_fine_reference(HOLDOUT, FINE_RESOLUTION_DISP_ENERGY_REACTION, N_SAMPLES, device, dtype)
print(f'43,400-element solution generated in {time.time() - t0:.1f}s.')

print(f'\n{"=" * 90}')
print('=== Loading the already-solved 6,840-element solution from dataset.h5 (no new FEM solve) ===')
coarse_ref = load_coarse_reference(HOLDOUT, N_SAMPLES, device, dtype)

print(f'\n{"=" * 90}')
print('=== FEM-vs-FEM discretization error: 6,840-el and 43,400-el solutions vs. the 950,400-el reference ===')
print('=== (NO neural network involved -- isolates pure mesh-discretization error) ===')
disc = compute_fem_discretization_errors(coarse_ref, mid_ref, fine_ref, device, dtype)

print(f'\n{"=" * 90}')
print('=== SUMMARY: every checkpoint vs. the single 950,400-element reference ===')
for name, r in results.items():
    s = r['summary']
    if 'mean_region_cauchy_field_rel_fine' in s:  # u-only path
        print(f'  {name}:')
        print(f'    disp_rel_l2 combined:        {s["mean_disp_rel_l2_fine_combined"]:.4f}')
        print(f'    H1_seminorm_rel:             {s["mean_H1_seminorm_rel_fine"]:.4f}')
        print(f'    energy rel err:              mean={s["mean_rel_err_energy_fine"]:.4f}')
        print(f'    reaction_moment_y rel err:   mean={s["mean_rel_err_reaction_moment_y_fine"]:.4f}')
        print(f'    region_cauchy_field_rel:     mean={s["mean_region_cauchy_field_rel_fine"]:.4f}')
        print(f'    region sigma components:    xx={s["mean_region_sigma_xx_rel_fine"]:.4f}  '
              f'yy={s["mean_region_sigma_yy_rel_fine"]:.4f}  zz={s["mean_region_sigma_zz_rel_fine"]:.4f}  '
              f'xy={s["mean_region_sigma_xy_rel_fine"]:.4f}  xz={s["mean_region_sigma_xz_rel_fine"]:.4f}  '
              f'yz={s["mean_region_sigma_yz_rel_fine"]:.4f}')
    else:  # mixed-stress path
        print(f'  {name}:')
        print(f'    H1_seminorm_rel:                     {s["mean_H1_seminorm_rel_fine"]:.4f}')
        print(f'    region_cauchy_field_rel IMPLICIT:    {s["mean_region_cauchy_field_rel_implicit_fine"]:.4f}')
        print(f'    region_cauchy_field_rel DIRECT:      {s["mean_region_cauchy_field_rel_direct_fine"]:.4f}')

print(f'\n{"=" * 90}')
print('=== SUMMARY: FEM-vs-FEM discretization error (no NN) vs. the 950,400-element reference ===')
for tag, label in (('vs_6840', '6,840-element'), ('vs_43400', '43,400-element')):
    s = disc[tag]['summary']
    print(f'  {label} solution:')
    print(f'    disp_rel_l2:               {s["mean_disp_rel_l2_vs_950400"]:.4f}')
    print(f'    H1_seminorm_rel:           {s["mean_H1_seminorm_rel_vs_950400"]:.4f}')
    print(f'    energy_rel:                {s["mean_energy_rel_vs_950400"]:.4f}')
    print(f'    reaction_moment_y_rel:     {s["mean_reaction_moment_y_rel_vs_950400"]:.4f}')
    print(f'    region_cauchy_field_rel:   {s["mean_region_cauchy_field_rel_vs_950400"]:.4f}')

try:
    from omar_pfem.run_manifest import write_manifest
    write_manifest(f'{R}/b3_rabczuk_reeval_v2', kind='b3_rabczuk_reeval_v2',
                    args={'fine_resolution': list(FINE_RESOLUTION_STRESS),
                          'mid_resolution_for_discretization_report': list(FINE_RESOLUTION_DISP_ENERGY_REACTION),
                          'n_samples': N_SAMPLES,
                          'checkpoints': [n for n, _ in U_ONLY_CHECKPOINTS + MIXED_STRESS_CHECKPOINTS]},
                    started_at=_started,
                    results={'elapsed_total_s': time.time() - _started},
                    outputs=[],
                    notes="Professor Rabczuk's second review: one common 950,400-element "
                          "reference for every QoI (not split by QoI as before), plus a new "
                          "FEM-vs-FEM discretization-error report separating mesh error from "
                          "neural-operator error. No retraining -- evaluation of existing "
                          "checkpoints only, per his own explicit instruction.")
except Exception as e:
    print(f'[manifest] not recorded: {e}')

print(f'\n{"=" * 90}')
print(f'Total wall-clock: {time.time() - _started:.1f}s')
print('\nDone. Send this cell\'s full printed output back -- both summary sections above.')
